# 6주차 실습: Vector Add → Fused Softmax

[학습 자료](https://github.com/Jake-Song/tensor2silicon/blob/main/week6/README.md)

GPU: Colab·Kaggle에서 GPU 런타임을 선택하고 순서대로 실행한다. 기존 PyTorch 환경을 사용한다. `triton` import가 실패하면 설치된 PyTorch가 요구하는 Triton 버전을 설치한다.

CPU: 아래 `INTERPRET=True`로 설정한다. PyTorch·Triton·NumPy가 설치된 Linux 환경에서 작은 입력만 확인하며 벤치마크는 건너뛴다. 설정 변경 시 런타임을 재시작한다.

첫 손계산 셀은 Python 표준 라이브러리만으로 실행할 수 있다. GPU 실측 출력은 미리 넣지 않았다.

In [ ]:
import math

N, BLOCK_SIZE = 10, 4
for pid in range((N + BLOCK_SIZE - 1) // BLOCK_SIZE):
    offsets = [pid * BLOCK_SIZE + i for i in range(BLOCK_SIZE)]
    print("pid", pid, "offsets", offsets, "mask", [i < N for i in offsets])

M, N, element_bytes = 4096, 1024, 4
q_naive = (8 * M * N + 4 * M) * element_bytes
q_fused = 2 * M * N * element_bytes
print("naive / fused Byte:", q_naive, q_fused)
print("memory ms:", q_naive / 1e12 * 1e3, q_fused / 1e12 * 1e3)
print("memory speedup:", q_naive / q_fused)
bad_denom = 781 * math.exp(-2) + 243
print("padding=0: denominator / valid row sum:", bad_denom, 781 * math.exp(-2) / bad_denom)

## 1. 환경

`TRITON_INTERPRET`는 Triton을 import하기 전에 설정한다.

In [ ]:
import os

INTERPRET = False  # CPU 해석: True. 바꾸면 런타임 재시작 후 처음부터 실행
WITH_COMPILE = False  # GPU에서 torch.compile 비교를 추가하려면 True
os.environ["TRITON_INTERPRET"] = "1" if INTERPRET else "0"

import json
import platform
import sys
import torch
import triton
import triton.language as tl
import triton.testing

if not INTERPRET and not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임을 선택하거나 재시작 후 INTERPRET=True로 실행하세요.")

device = torch.device("cpu" if INTERPRET else "cuda")
torch.manual_seed(0)
env_info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "triton": triton.__version__,
    "cuda_build": torch.version.cuda,
    "gpu": None if INTERPRET else torch.cuda.get_device_name(device),
    "mode": "CPU interpreter" if INTERPRET else "GPU",
}
print(json.dumps(env_info, ensure_ascii=False, indent=2))

## 2. Vector Add

`pid → offsets → mask → load/store`를 코드에서 찾는다. `BLOCK_SIZE`는 원소 수다.

[공식 Tutorial 01: Compute Kernel](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html#compute-kernel)

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def add(x, y, block_size=4):
    assert x.ndim == y.ndim == 1 and x.shape == y.shape
    assert x.is_contiguous() and y.is_contiguous()
    assert x.dtype == y.dtype == torch.float32 and x.device == y.device
    assert block_size > 0 and block_size & (block_size - 1) == 0
    out = torch.empty_like(x)
    add_kernel[(triton.cdiv(x.numel(), block_size),)](
        x, y, out, x.numel(), BLOCK_SIZE=block_size, num_warps=4
    )
    return out


a = torch.arange(10, device=device, dtype=torch.float32)
b = torch.ones_like(a)
actual = add(a, b)
assert torch.allclose(actual, a + b)
print("N=10, BLOCK_SIZE=4, grid=(3,):", actual.tolist())

## 3. 한 프로그램이 한 행을 처리하는 Softmax

학습용으로 `grid=(M,)`를 사용한다. 공식 최신 예제의 persistent loop·occupancy 계산은 생략했다. 입력은 유한한 FP32, 열 stride는 1이다. `PADDING`은 잘못된 패딩 실험을 위한 상수다.

[공식 Tutorial 02: Compute Kernel](https://triton-lang.org/main/getting-started/tutorials/02-fused-softmax.html#compute-kernel)

In [ ]:
def naive_softmax(x):
    x_max = x.max(dim=1)[0]
    z = x - x_max[:, None]
    numerator = torch.exp(z)
    denominator = numerator.sum(dim=1)
    return numerator / denominator[:, None]


@triton.jit
def softmax_kernel(
    input_ptr, output_ptr, input_row_stride, output_row_stride, n_cols,
    BLOCK_SIZE: tl.constexpr, PADDING: tl.constexpr,
):
    row_idx = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    row_start_ptr = input_ptr + row_idx * input_row_stride
    row = tl.load(row_start_ptr + cols, mask=mask, other=PADDING)
    numerator = tl.exp(row - tl.max(row, axis=0))
    result = numerator / tl.sum(numerator, axis=0)
    output_ptrs = output_ptr + row_idx * output_row_stride + cols
    tl.store(output_ptrs, result, mask=mask)


def softmax(x, padding=-float("inf")):
    assert x.ndim == 2 and x.dtype == torch.float32
    assert x.shape[0] > 0 and x.shape[1] > 0 and x.stride(1) == 1
    m, n = x.shape
    out = torch.empty((m, n), device=x.device, dtype=x.dtype)
    softmax_kernel[(m,)](
        x, out, x.stride(0), out.stride(0), n,
        BLOCK_SIZE=triton.next_power_of_2(n), PADDING=padding, num_warps=4,
    )
    return out

## 4. 정확도와 잘못된 패딩

GPU에서는 `[1823,781]`, CPU 해석에서는 `[3,781]`로 검사한다. 781열은 마스크를, 음수 행은 패딩값을 검사한다. `allclose`는 비트 일치 검사가 아니다. `tl.exp` 근사와 리덕션 순서로 작은 오차가 생길 수 있다.

In [ ]:
RTOL, ATOL = 1e-5, 1e-8
correctness = []
rows = 3 if INTERPRET else 1823

for case in ("random", "negative", "row_stride"):
    if case == "random":
        x = torch.randn((rows, 781), device=device, dtype=torch.float32)
    elif case == "negative":
        x = torch.full((3, 781), -2.0, device=device, dtype=torch.float32)
    else:
        x = torch.randn((rows * 2, 781), device=device, dtype=torch.float32)[::2]
    reference = torch.softmax(x, dim=-1)
    for name, fn in (("naive", naive_softmax), ("triton", softmax)):
        actual = fn(x)
        ok = torch.allclose(actual, reference, rtol=RTOL, atol=ATOL)
        assert ok, (case, name, (actual - reference).abs().max().item())
        assert torch.allclose(actual.sum(dim=1), torch.ones(x.shape[0], device=device), rtol=RTOL, atol=ATOL)
        correctness.append({"case": case, "provider": name, "shape": list(x.shape), "allclose": ok})
print("정확도 검사 통과:", correctness)

x_negative = torch.full((1, 781), -2.0, device=device, dtype=torch.float32)
good = softmax(x_negative)
bad = softmax(x_negative, padding=0.0)
expected_bad_sum = 781 * math.exp(-2) / (781 * math.exp(-2) + 243)
assert not torch.allclose(bad, good, rtol=RTOL, atol=ATOL)
assert math.isclose(bad.sum().item(), expected_bad_sum, rel_tol=1e-5)
print("유효 출력 합: -inf 패딩", good.sum().item(), "/ 0 패딩", bad.sum().item())

## 5. GPU 벤치마크

입력 `[4096,1024]`, FP32. 각 함수는 출력 할당을 포함한다. 같은 입력으로 준비 실행 후 동기화하고 GPU event 구간을 측정한다. 컴파일·입력 생성·전송·정확도 검사는 제외한다. event 사이의 커널 제출 간격이 포함될 수 있다.

`warmup=25`, `rep=100`은 **ms 단위 시간 예산**이다. 모든 구현의 실효 GB/s 분자는 필수 입출력 `2MN×4 Byte`로 통일한다. naive의 실제 HBM 이동량을 잰 값은 아니다. 손계산 모델은 max의 인덱스 출력·임시 버퍼를 생략한다. `torch.compile`의 fusion 여부와 속도는 실행 환경에 따라 달라진다.

[do_bench API](https://triton-lang.org/main/python-api/generated/triton.testing.do_bench.html) · [PyTorch compiler](https://docs.pytorch.org/docs/stable/torch.compiler.html)

In [ ]:
benchmark_results = []
if INTERPRET:
    print("CPU 해석 모드: 성능 측정 생략")
else:
    M, N = 4096, 1024
    x_bench = torch.randn((M, N), device=device, dtype=torch.float32)
    providers = {
        "naive": naive_softmax,
        "torch.softmax": lambda x: torch.softmax(x, dim=-1),
        "triton": softmax,
    }
    if WITH_COMPILE:
        providers["torch.compile(naive)"] = torch.compile(naive_softmax)

    reference = torch.softmax(x_bench, dim=-1)
    useful_bytes = 2 * M * N * x_bench.element_size()
    for name, fn in providers.items():
        # 이 shape의 JIT/torch.compile 및 준비 실행은 측정 전에 완료
        for _ in range(3):
            prepared = fn(x_bench)
        assert torch.allclose(prepared, reference, rtol=RTOL, atol=ATOL), name
        torch.cuda.synchronize()
        ms = triton.testing.do_bench(
            lambda: fn(x_bench), warmup=25, rep=100, return_mode="median"
        )
        gbps = useful_bytes / (ms * 1e-3) / 1e9
        benchmark_results.append({"provider": name, "ms_median": ms, "effective_GB_s": gbps})
        print(f"{name:24s} {ms:.6f} ms  {gbps:.2f} GB/s")

    record = {
        "environment": env_info,
        "shape": [M, N], "dtype": "float32", "BLOCK_SIZE": 1024, "num_warps": 4,
        "measurement": "do_bench GPU event interval; output allocation in callable",
        "warmup_ms": 25, "rep_ms": 100, "statistic": "median",
        "useful_bytes": useful_bytes, "rtol": RTOL, "atol": ATOL,
        "correctness": correctness, "results": benchmark_results,
    }
    from pathlib import Path
    Path("week6_results.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print("환경·정확도·측정 결과 저장: week6_results.json")

## 설명해보기

1. `pid=2`의 `offsets=[8,9,10,11]`에서 실제 읽고 쓰는 주소는?
2. `-inf` 대신 0을 채우면 왜 store 마스크로 해결되지 않는가?
3. 메모리 시간 약 4배와 실제 속도 향상은 왜 다를 수 있는가?

CPU 해석 중에는 `add_kernel`의 `mask` 다음 줄에 `print(offsets)`·`print(mask)` 또는 `pdb.set_trace()`를 넣어 관찰할 수 있다. NumPy 기반 순차 해석이므로 시간 비교에 쓰지 않는다. bfloat16과 간접 포인터 접근에는 제약이 있다.

[공식 Debugging Guide](https://triton-lang.org/main/programming-guide/chapter-3/debugging.html#using-the-interpreter)